# Del texto a los vectores

_Cuatro épocas del NLP resolviendo el mismo problema_

Este cuaderno acompaña al capítulo [Del texto a los vectores](https://iraitzm.github.io/manual-ia-generativa/parts/fundamentos/nlp.html) del manual.

La diferencia con la mayoría de tutoriales de NLP es que aquí **no cambiamos de ejemplo en cada sección**. Hay un solo problema, el mismo de principio a fin, y lo resolvemos cuatro veces con las herramientas de cuatro épocas distintas. Al final comparamos acierto y latencia de las cuatro.

El problema: una secretaría académica recibe consultas por escrito y quiere enrutarlas al departamento que corresponde.

| Sección | Época | Herramienta |
|---|---|---|
| 2 | 1950-1990 | Reglas escritas a mano |
| 3 | 1990-2013 | Recuento de palabras y TF-IDF |
| 4 | 2013-2018 | Embeddings de frase |
| 5 | 2020- | Un modelo generativo con instrucciones |

_Lo que vamos a hacer_

> **Cómo ejecutarlo**
>
> En Google Colab funciona tal cual, con GPU recomendada para la sección 5. En local hace falta un entorno con Python 3.12 o superior.


## Preparación

Todo lo que necesita el cuaderno se instala aquí. Es la única celda de instalación: si algo falla más adelante, el problema no es una dependencia que falte.

In [ ]:
%pip install -q spacy scikit-learn sentence-transformers "transformers>=4.51" torch matplotlib pandas
!python -m spacy download es_core_news_sm

In [ ]:
import time
import numpy as np

SEMILLA = 42
np.random.seed(SEMILLA)

Fijar la semilla no es un detalle cosmético. Sin ella, cada ejecución da números distintos y no se puede saber si una mejora viene de lo que hemos cambiado o del azar del reparto de datos.

### Los datos

Un conjunto pequeño y escrito a mano, para que se pueda leer entero y entender por qué falla cada método. Seis categorías y ocho consultas por categoría.

In [ ]:
CONSULTAS = [
    # --- MATRICULA ---------------------------------------------------
    ("quiero matricularme en el máster de análisis de datos", "MATRICULA"),
    ("cuándo se abre el plazo de matrícula del curso que viene", "MATRICULA"),
    ("puedo añadir una asignatura optativa a mi matrícula", "MATRICULA"),
    ("necesito cambiar de grupo en la asignatura de Estadística", "MATRICULA"),
    ("cuántos créditos tengo que coger como mínimo", "MATRICULA"),
    ("me he equivocado al elegir asignaturas, se puede modificar", "MATRICULA"),
    ("hay lista de espera para entrar en el itinerario de datos", "MATRICULA"),
    ("qué documentación hace falta para formalizar la inscripción", "MATRICULA"),

    # --- BAJAS -------------------------------------------------------
    ("quiero anular la matrícula de Estadística", "BAJAS"),
    ("querría darme de baja en dos asignaturas", "BAJAS"),
    ("he decidido no continuar con el grado este año", "BAJAS"),
    ("cómo dejo la carrera sin perder la plaza", "BAJAS"),
    ("puedo renunciar a una convocatoria de examen", "BAJAS"),
    ("solicito el traslado de expediente a otra universidad", "BAJAS"),
    ("si abandono ahora me devuelven el dinero", "BAJAS"),
    ("cuál es el plazo para desistir de la inscripción", "BAJAS"),

    # --- BECAS -------------------------------------------------------
    ("cómo solicito la beca de comedor", "BECAS"),
    ("plazo para pedir la beca general del ministerio", "BECAS"),
    ("me han denegado la ayuda al estudio y quiero recurrir", "BECAS"),
    ("qué renta máxima admite la convocatoria de ayudas", "BECAS"),
    ("cuándo cobraré el importe de la beca concedida", "BECAS"),
    ("puedo compaginar dos ayudas al mismo tiempo", "BECAS"),
    ("necesito el justificante de que soy becario", "BECAS"),
    ("perdí la beca por suspender, puedo recuperarla", "BECAS"),

    # --- ACTAS -------------------------------------------------------
    ("no me aparece la nota de Estadística en el expediente", "ACTAS"),
    ("creo que hay un error en mi calificación de Álgebra", "ACTAS"),
    ("cuándo se publican las notas de la convocatoria de enero", "ACTAS"),
    ("quiero revisar el examen con el profesor", "ACTAS"),
    ("mi asignatura sigue como no presentado y sí me presenté", "ACTAS"),
    ("cómo reclamo una calificación con la que no estoy de acuerdo", "ACTAS"),
    ("me falta por convalidar una asignatura del año pasado", "ACTAS"),
    ("la media del expediente no me cuadra con mis notas", "ACTAS"),

    # --- CERTIFICADOS ------------------------------------------------
    ("necesito un certificado de notas para una empresa", "CERTIFICADOS"),
    ("cómo pido el suplemento europeo al título", "CERTIFICADOS"),
    ("quiero un justificante de que estoy matriculado", "CERTIFICADOS"),
    ("dónde recojo el título una vez expedido", "CERTIFICADOS"),
    ("me hace falta un documento acreditativo del nivel de inglés", "CERTIFICADOS"),
    ("puedo descargar el expediente académico en pdf", "CERTIFICADOS"),
    ("necesito una copia compulsada del título de grado", "CERTIFICADOS"),
    ("cuánto tarda en emitirse el certificado académico personal", "CERTIFICADOS"),

    # --- PRACTICAS ---------------------------------------------------
    ("cuándo empiezan las prácticas en empresa", "PRACTICAS"),
    ("puedo hacer las prácticas en la empresa donde ya trabajo", "PRACTICAS"),
    ("quién es mi tutor académico de prácticas curriculares", "PRACTICAS"),
    ("cuántas horas de estancia hay que cumplir", "PRACTICAS"),
    ("hay convenio firmado con esta consultora", "PRACTICAS"),
    ("me convalidan la experiencia laboral por las prácticas", "PRACTICAS"),
    ("dónde subo la memoria final del periodo en empresa", "PRACTICAS"),
    ("puedo hacer prácticas extracurriculares en verano", "PRACTICAS"),
]

textos = [t for t, _ in CONSULTAS]
etiquetas = [e for _, e in CONSULTAS]

print(f"{len(textos)} consultas, {len(set(etiquetas))} categorías")

Un reparto estratificado, para que las cuatro épocas se midan exactamente sobre las mismas consultas.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    textos, etiquetas, test_size=0.25, stratify=etiquetas, random_state=SEMILLA
)

print(f"{len(X_train)} para entrenar, {len(X_test)} para evaluar")

> **Doce consultas de evaluación no miden nada**
>
> Con un conjunto tan pequeño, una diferencia de dos aciertos cambia la métrica un 16%. Los números de este cuaderno sirven para **ver la forma del resultado**, no para decidir nada.
>
> Cualquier comparación seria necesita cientos de ejemplos y validación cruzada. Aquí cabe todo en pantalla, que es lo que buscamos.


Y una función común para medir, porque comparar cuatro cosas exige medirlas igual.

In [ ]:
from sklearn.metrics import f1_score

resultados = []

def evaluar(nombre, predecir, entradas=X_test, esperado=y_test):
    """Aplica `predecir` a cada entrada y anota acierto y latencia."""
    inicio = time.perf_counter()
    predicho = [predecir(t) for t in entradas]
    transcurrido = time.perf_counter() - inicio

    f1 = f1_score(esperado, predicho, average="macro", zero_division=0)
    ms = 1000 * transcurrido / len(entradas)
    resultados.append({"enfoque": nombre, "f1_macro": round(f1, 3), "ms_por_consulta": round(ms, 2)})

    print(f"{nombre:28} F1 ={f1:.3f}   {ms:7.2f} ms/consulta")
    return predicho

## Primera generación: reglas escritas a mano

Sin datos de entrenamiento, sin modelo y sin dependencias. Solo alguien de la secretaría escribiendo lo que sabe.

In [ ]:
REGLAS = [
    (("anular", "baja", "abandon", "renunci", "traslado"), "BAJAS"),
    (("beca", "ayuda", "becari"), "BECAS"),
    (("nota", "calificaci", "acta", "examen", "revisi"), "ACTAS"),
    (("certificad", "justificante", "título", "expediente académico"), "CERTIFICADOS"),
    (("práctica", "prácticas", "empresa", "convenio", "tutor"), "PRACTICAS"),
    (("matricul", "matrícula", "asignatura", "crédito", "inscripción"), "MATRICULA"),
]

def clasificar_con_reglas(texto):
    minusculas = texto.lower()
    for patrones, categoria in REGLAS:
        if any(p in minusculas for p in patrones):
            return categoria
    return "MATRICULA"  # categoría por defecto: la más frecuente

predicho_reglas = evaluar("1. Reglas", clasificar_con_reglas)

El orden de las reglas importa, y mucho. "Quiero anular la matrícula" contiene tanto `anular` como `matrícula`, así que la respuesta depende de cuál se comprueba antes. Ese es el primer síntoma de que el enfoque no escala: cada regla nueva puede romper una vieja, y no hay forma de saberlo sin volver a probarlo todo.

In [ ]:
for texto, esperado, obtenido in zip(X_test, y_test, predicho_reglas):
    if esperado != obtenido:
        print(f"  {esperado:13} → {obtenido:13} | {texto}")

l mantenimiento manual y la necesidad constante e finamiento de las reglas es lo que supuso el fin del enfoque.

## Segunda generación: contar palabras

Antes de contar hay que decidir qué se cuenta. Veamos qué hace un analizador lingüístico con una consulta.

In [ ]:
import spacy

nlp = spacy.load("es_core_news_sm")
doc = nlp("Querría anular la matrícula de dos asignaturas antes del viernes")

print(f"{'token':<14}{'lema':<14}{'categoría':<12}{'vacía'}")
for token in doc:
    print(f"{token.text:<14}{token.lemma_:<14}{token.pos_:<12}{token.is_stop}")

Tres cosas que mirar en esa tabla. La **lematización** lleva "querría" a `querer` y "asignaturas" a `asignatura`, de modo que todas las formas de una palabra cuentan como la misma. Las **palabras vacías** marcadas como `True` son las que aparecen en todo y no distinguen nada. Y la **categoría gramatical** permitiría quedarse solo con verbos y sustantivos, que es donde está el contenido.

In [ ]:
def normalizar(texto):
    """Lemas en minúscula de las palabras con contenido."""
    return " ".join(
        token.lemma_.lower()
        for token in nlp(texto)
        if not token.is_stop and not token.is_punct and token.pos_ in {"NOUN", "VERB", "ADJ", "PROPN"}
    )

print(normalizar("Querría anular la matrícula de dos asignaturas antes del viernes"))

Ahora sí, TF-IDF. Cada consulta pasa a ser un vector con una posición por término del vocabulario, y cada posición pesa la frecuencia del término por lo raro que es en el corpus.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

tfidf = make_pipeline(
    TfidfVectorizer(preprocessor=normalizar, ngram_range=(1, 2)),
    LogisticRegression(max_iter=1000, random_state=SEMILLA),
)
tfidf.fit(X_train, y_train)

evaluar("2. TF-IDF + regresión", lambda t: tfidf.predict([t])[0])

Miremos el vocabulario que ha construido y lo que pesa cada término, que es donde se ve cómo funciona por dentro.

In [ ]:
vectorizador = tfidf.named_steps["tfidfvectorizer"]
vocabulario = vectorizador.get_feature_names_out()

print(f"{len(vocabulario)} términos en el vocabulario")
print(f"Cada consulta es un vector de {len(vocabulario)} dimensiones\n")

fila = vectorizador.transform(["quiero anular la matrícula de Estadística"]).toarray()[0]
activos = np.argsort(fila)[::-1][:5]
for i in activos:
    if fila[i] > 0:
        print(f"  {vocabulario[i]:<25} {fila[i]:.3f}")

print(f"\nDimensiones distintas de cero: {(fila > 0).sum()} de {len(vocabulario)}")

Esa última línea es la limitación estructural del método: **el vector es enorme y está casi todo a cero**. El significado no está en el vector, está repartido entre miles de posiciones vacías.

### El agujero que no se puede tapar

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

pares = [
    ("quiero anular la matrícula", "querría darme de baja"),
    ("quiero anular la matrícula", "plazo para pedir la beca"),
]

for a, b in pares:
    va, vb = vectorizador.transform([a]), vectorizador.transform([b])
    print(f"  {cosine_similarity(va, vb)[0][0]:.3f}   {a!r} vs {b!r}")

Las dos primeras frases significan lo mismo y no comparten ninguna palabra de contenido, así que su similitud es cero. Para TF-IDF son tan distintas como cualquier par al azar. Esto es la **sinonimia**, y no tiene solución dentro del paradigma: se podría meter una lista de sinónimos a mano, y entonces estaríamos de vuelta en la época 1.

## Tercera generación: embeddings

Un modelo entrenado para que frases con el mismo significado caigan cerca en el espacio, aunque no compartan una sola palabra.

In [ ]:
from sentence_transformers import SentenceTransformer

codificador = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

E_train = codificador.encode(X_train, show_progress_bar=False)
E_test = codificador.encode(X_test, show_progress_bar=False)

print(f"Cada consulta es ahora un vector de {E_train.shape[1]} dimensiones, todas con valor")

Trescientas ochenta y cuatro dimensiones densas frente a los cientos de dimensiones vacías de TF-IDF. Y con el mismo par de antes:

In [ ]:
for a, b in pares:
    va, vb = codificador.encode([a]), codificador.encode([b])
    print(f"  {cosine_similarity(va, vb)[0][0]:.3f}   {a!r} vs {b!r}")

El par que TF-IDF daba por ortogonal sale ahora con una similitud alta. Eso, y nada más que eso, es lo que separa la búsqueda por palabras de la búsqueda semántica.

In [ ]:
clasificador = LogisticRegression(max_iter=1000, random_state=SEMILLA)
clasificador.fit(E_train, y_train)

evaluar("3. Embeddings + regresión", lambda t: clasificador.predict(codificador.encode([t]))[0])

### Ver el espacio

Trescientas ochenta y cuatro dimensiones no se dibujan, pero se pueden proyectar a dos quedándose con las direcciones de mayor variación.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

proyeccion = PCA(n_components=2, random_state=SEMILLA).fit_transform(codificador.encode(textos))

plt.figure(figsize=(9, 6))
for categoria in sorted(set(etiquetas)):
    indices = [i for i, e in enumerate(etiquetas) if e == categoria]
    plt.scatter(proyeccion[indices, 0], proyeccion[indices, 1], label=categoria, s=60, alpha=0.75)

plt.legend()
plt.title("Las 48 consultas proyectadas a dos dimensiones")
plt.tight_layout()
plt.show()

Nadie le ha dicho al modelo cuáles son las categorías. Los grupos que se ven salen solo de que el modelo aprendió, leyendo texto, que ciertas frases se usan en contextos parecidos. Fijaos también en qué categorías se solapan: ahí es donde fallará el clasificador, y suele coincidir con los pares que a una persona también le costaría separar.

### Búsqueda semántica

El mismo espacio sirve para buscar, que es exactamente lo que hace un sistema de recuperación.

In [ ]:
E_todos = codificador.encode(textos, show_progress_bar=False)

def buscar(pregunta, k=3):
    similitudes = cosine_similarity(codificador.encode([pregunta]), E_todos)[0]
    for i in np.argsort(similitudes)[::-1][:k]:
        print(f"  {similitudes[i]:.3f}  [{etiquetas[i]:<12}] {textos[i]}")

buscar("me quiero borrar de una asignatura")

Ni "borrar" ni "asignatura" aparecen juntas en ninguna consulta del conjunto, y aun así lo primero que devuelve es lo correcto. Este bloque de seis líneas es, en esencia, la pieza de recuperación de cualquier RAG.

## Cuarta generación: un modelo generativo

Ahora sin entrenar nada: se le explica la tarea al modelo por escrito y se le pide que responda.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

NOMBRE = "Qwen/Qwen3-0.6B"
tokenizador = AutoTokenizer.from_pretrained(NOMBRE)
modelo = AutoModelForCausalLM.from_pretrained(NOMBRE, torch_dtype="auto", device_map="auto")

print(f"{sum(p.numel() for p in modelo.parameters()) / 1e6:.0f} millones de parámetros")

> **Por qué un modelo tan pequeño**
>
> Un modelo de 600 millones de parámetros da resultados mediocres comparado con cualquier asistente comercial, y eso es deliberado: cabe en la memoria gratuita de Colab y permite ver el mecanismo. Si tenéis acceso a un modelo mayor, la comparación final sale más favorable a esta época y las conclusiones no cambian.

In [ ]:
CATEGORIAS = sorted(set(etiquetas))

INSTRUCCIONES = f"""Eres el sistema de enrutado de una secretaría académica.
Clasifica la consulta del alumno en exactamente una de estas categorías:
{", ".join(CATEGORIAS)}.
Responde únicamente con el nombre de la categoría, sin explicaciones."""

def preguntar(mensajes, max_tokens=16, temperatura=0.0):
    entrada = tokenizador.apply_chat_template(
        mensajes, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    tokens = tokenizador([entrada], return_tensors="pt").to(modelo.device)

    with torch.no_grad():
        salida = modelo.generate(
            **tokens,
            max_new_tokens=max_tokens,
            do_sample=temperatura > 0,
            temperature=temperatura if temperatura > 0 else None,
            pad_token_id=tokenizador.eos_token_id,
        )

    nuevos = salida[0][tokens.input_ids.shape[1]:]
    return tokenizador.decode(nuevos, skip_special_tokens=True).strip()

print(preguntar([
    {"role": "system", "content": INSTRUCCIONES},
    {"role": "user", "content": "quiero anular la matrícula de Estadística"},
]))

`enable_thinking=False` desactiva el modo de razonamiento de Qwen3. Con él activado el modelo escribiría su cadena de pensamiento antes de responder, lo que para clasificar en seis categorías es gastar tokens sin ganar nada.

La salida es texto libre, así que hay que normalizarla antes de compararla. Esto es trabajo que los tres enfoques anteriores no daban: **un clasificador devuelve una etiqueta, un generador devuelve una frase que hay que interpretar**.

In [ ]:
def clasificar_con_llm(texto):
    respuesta = preguntar([
        {"role": "system", "content": INSTRUCCIONES},
        {"role": "user", "content": texto},
    ]).upper()

    for categoria in CATEGORIAS:
        if categoria in respuesta:
            return categoria
    return "MATRICULA"  # no dijo ninguna categoría válida

evaluar("4. Generativo (zero-shot)", clasificar_con_llm)

Dos preguntas para pensar. La primera: ¿qué temperatura querríais para enrutar consultas, y qué temperatura para redactar la respuesta al alumno? La segunda, más incómoda: con temperatura 0 el modelo es determinista, pero **no es estable en el tiempo**, porque el proveedor puede actualizar el modelo detrás del mismo nombre. ¿Qué implica eso para un sistema en producción?

## La comparación

In [ ]:
import pandas as pd

tabla = pd.DataFrame(resultados)
tabla["veces_más_lento"] = (tabla.ms_por_consulta / tabla.ms_por_consulta.min()).round(0)
tabla

Los números concretos dependen de la máquina y del reparto de datos, pero la forma se repite siempre y es lo único que hay que llevarse:

* **Del primero al segundo enfoque el salto de acierto es grande.** Dejar de escribir reglas y empezar a aprender de ejemplos es la mejora más rentable de la tabla.
* **Del segundo al tercero el salto es menor pero real**, y viene casi todo de resolver la sinonimia.
* **Del tercero al cuarto el acierto sube poco o baja**, y la latencia se multiplica por cientos.

Esa última línea es el resultado importante del cuaderno. Para una tarea cerrada, con etiquetas estables y volumen alto, **el modelo generativo es la herramienta más cara y más lenta**, y solo compensa cuando hay que hacer algo que las otras no pueden: redactar la respuesta, tratar categorías que cambian cada semana o resolver casos que nadie enumeró.

## Para seguir

* [De la convolución a la atención](https://iraitzm.github.io/manual-ia-generativa/parts/fundamentos/redes.html), el capítulo que explica por qué la arquitectura de la sección 4 es como es.
* [Un transformer actual, pieza a pieza](https://colab.research.google.com/github/IraitzM/manual-ia-generativa/blob/main/notebooks/modelos/qwen-desde-cero.ipynb), el cuaderno que monta ese modelo por dentro.
* [Recuperación](https://iraitzm.github.io/manual-ia-generativa/parts/contexto/rag.html), donde la búsqueda semántica de la sección 3 pasa a ser un sistema completo.